# T5: who answers for the heat, and a notice they can be sent

Runs on the team's IBM z17 LinuxONE guest, in the `heatahead-jupyter` kernel, from the
`~/heatahead/site` clone (task: `docs/tasks/T5-messaging-landlord.md`).

1. Pull the managing-agent and corporate-owner rows of HPD registration contacts
   (NYC Open Data `feu5-w2e2`) and join them to the owner already in `bbl_landlord.csv`
   (`web/build_contacts.py`).
2. Check the privacy rules: business addresses only, street addresses only for companies,
   no unit or apartment designator anywhere.
3. Rebuild `public/data/` for the Vercel site with the new fields (`web/build_assets.py`).
4. Draft the tenant notice and the landlord self-audit for every ranked building through
   the same code the site serves, and time it on s390x.

`hpd_contacts.csv` and `bbl_landlord.csv` feed the frozen model's portfolio features, so
they are read here and never rewritten. Nothing in this notebook sends a message.

In [1]:
import os, platform, subprocess, sys, time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "web"))
import pandas as pd

print("machine :", platform.machine(), "|", platform.node())
print("python  :", platform.python_version(), "| pandas", pd.__version__)
print("repo    :", ROOT, "| git", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())

machine : s390x | datathon
python  : 3.12.3 | pandas 2.1.4
repo    : /home/linux1/heatahead/site | git 390ca9b


## 1. Pull HPD registration contacts and join them to the owner of record

In [2]:
import build_contacts as bc

t0 = time.time()
raw = bc.pull()                      # Agent + CorporateOwner rows, paged 50,000 at a time
contacts = bc.build(raw)
print(f"pull + join: {time.time() - t0:.1f} s on {platform.machine()}")
print("columns pulled:", list(raw.columns))
contacts[contacts.bbl == 2043190010].T   # Building 60119, the running example

  feu5-w2e2 offset=0 rows=50,000
  feu5-w2e2 offset=50,000 rows=50,000
  feu5-w2e2 offset=100,000 rows=50,000
  feu5-w2e2 offset=150,000 rows=50,000
  feu5-w2e2 offset=200,000 rows=50,000
  feu5-w2e2 offset=250,000 rows=36,684
pulled 286,684 contact rows in 26 s -> hpd_contacts_business.csv
pull + join: 31.8 s on s390x
columns pulled: ['registrationcontactid', 'registrationid', 'type', 'corporationname', 'firstname', 'lastname', 'businesshousenumber', 'businessstreetname', 'businesscity', 'businessstate', 'businesszip']


,75891
bbl,2043190010
registrationid,206228
agent,"HSC MANAGEMENT CORP., AMO"
agent_co,1
agent_addr,"5804A Tyndall Avenue, Bronx, NY 10471"
owner_addr,"2156 Cruger Avenue, Bronx, NY 10462"
hpd_id,60119
reg_last,2025-10-08


## 2. Coverage on the 63,727 ranked buildings, and the privacy checks

A street address is shown only for a company: the managing agent's business address
when the agent files as a company, the owner's when the owner is a corporation. An agent
filed as a person shows a name and a city; the building's full registration stays one
click away on HPD Online.

In [3]:
risk = pd.read_csv("model/out/season_risk_2026.csv", usecols=["bbl", "rank", "p_season"])
c = contacts.fillna({"agent": "", "agent_addr": "", "owner_addr": ""})
m = risk.merge(c, on="bbl", how="left")
m["boro"] = (m.bbl // 1_000_000_000).map({1: "Manhattan", 2: "Bronx", 3: "Brooklyn",
                                         4: "Queens", 5: "Staten Island"})
reg = m.registrationid.notna()
cov = pd.DataFrame({
    "ranked buildings": m.groupby("boro").size(),
    "registered owner": m[reg].groupby("boro").size(),
    "managing agent": m[m.agent.fillna("") != ""].groupby("boro").size(),
    "agent street address (company)": m[m.agent_co == 1].groupby("boro").size(),
    "owner street address (corporation)": m[m.owner_addr.fillna("") != ""].groupby("boro").size(),
})
cov.loc["New York City"] = cov.sum()
share = (cov.iloc[:, 1:].div(cov["ranked buildings"], axis=0) * 100).round(1)
share.insert(0, "ranked buildings", cov["ranked buildings"])
share

,ranked buildings,registered owner,managing agent,agent street address (company),owner street address (corporation)
boro,,,,,
Bronx,12773,94.9,82.9,51.0,73.6
Brooklyn,24252,94.1,82.1,47.2,71.0
Manhattan,15345,99.0,97.4,76.3,95.3
Queens,10657,88.6,75.5,41.5,62.2
Staten Island,700,81.6,68.4,44.4,59.4
New York City,63727,94.4,84.7,54.0,75.8


In [4]:
unit = r"#|\b(?:apt|apartment|unit)\b"
checks = {
    "businessapartment column never pulled": "businessapartment" not in raw.columns,
    "no unit designator in an agent address": not c.agent_addr.str.contains(unit, case=False, regex=True).any(),
    "no unit designator in an owner address": not c.owner_addr.str.contains(unit, case=False, regex=True).any(),
    "agents filed as a person carry no street number": not c[(c.agent_co == 0) & (c.agent != "")]
        .agent_addr.str.contains(r"\d", regex=True).any(),
    "one row per bbl": c.bbl.is_unique,
    "bbl and registrationid are int64": str(c.bbl.dtype) == "int64" and str(c.registrationid.dtype) == "int64",
}
for k, v in checks.items():
    print("PASS" if v else "FAIL", k)
assert all(checks.values())

PASS businessapartment column never pulled
PASS no unit designator in an agent address
PASS no unit designator in an owner address
PASS agents filed as a person carry no street number
PASS one row per bbl
PASS bbl and registrationid are int64


How concentrated is responsibility? Among the ranked buildings with a modeled winter risk
of 50% or more, the share held by the largest managing agents (counts only; this notebook
does not publish a list of names).

In [5]:
hi = m[(m.p_season >= 0.5) & (m.agent.fillna("") != "")]
per = hi.groupby("agent").size().sort_values(ascending=False)
print(f"{len(m[m.p_season >= 0.5]):,} buildings at 50%+ risk; {len(hi):,} have a managing agent "
      f"on file, spread over {per.size:,} agents")
for k in (10, 50, 100):
    print(f"  the {k} agents with the most of them hold {per.head(k).sum():,} "
          f"({per.head(k).sum() / len(hi):.1%})")

8,914 buildings at 50%+ risk; 8,832 have a managing agent on file, spread over 3,893 agents
  the 10 agents with the most of them hold 730 (8.3%)
  the 50 agents with the most of them hold 1,890 (21.4%)
  the 100 agents with the most of them hold 2,616 (29.6%)


## 3. Rebuild `public/data/` for the Vercel site with the new fields

In [6]:
t0 = time.time()
r = subprocess.run([sys.executable, "web/build_assets.py"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0
print(f"build_assets: {time.time() - t0:.1f} s")

loading model outputs...
loaded: 63,727 ranked buildings
wrote 132 files, 19.4 MB total; biggest public/data/map_data.json 3.1 MB
{
 "median": 0.0758,
 "ranked": 63727,
 "cold_ranked": 67125,
 "at_least_80": 3332,
 "at_least_50": 8914,
 "shards": 128,
 "built_at": "2026-10-17T19:13:57+00:00",
 "built_on": "s390x",
 "git_sha": "390ca9b",
 "model_files": {
  "season_risk_2026.csv": {
   "bytes": 4786256,
   "modified": "2026-10-17T17:42:55+00:00"
  },
  "ranked_inspection_list.csv": {
   "bytes": 7361709,
   "modified": "2026-10-17T17:42:11+00:00"
  },
  "season_reliability.csv": {
   "bytes": 290,
   "modified": "2026-10-17T17:42:31+00:00"
  },
  "building_day_counts.csv": {
   "bytes": 27569022,
   "modified": "2026-10-17T15:58:03+00:00"
  },
  "heat_model_final.json": {
   "bytes": 2670386,
   "modified": "2026-10-17T17:42:06+00:00"
  },
  "calibration.csv": {
   "bytes": 4204262,
   "modified": "2026-10-17T16:01:09+00:00"
  },
  "bbl_contacts.csv": {
   "bytes": 17387265,
   "modifie

## 4. Every notice, drafted through the code the site serves

`app.building_payload()` and `app.draft_message()` are exactly what
`/api/draft_message/<bbl>` returns, in FULL mode as on this instance. Building 60119
(BBL 2043190010) is the running example in the submission.

In [7]:
os.environ["APP_MODE"] = "full"
import app

p = app.building_payload(2043190010)
print({k: p[k] for k in ("season", "last_season", "landlord", "agent", "registration")})

loading model outputs...
loaded: 63,727 ranked buildings
{'season': {'p': 0.99, 'rank': 323, 'of': 63727, 'vs_median': 13.1}, 'last_season': {'season': '2025-26', 'complaints': 127, 'violations': 0}, 'landlord': {'name': '2156 Cruger Avenue Apartment Corp', 'portfolio': 1, 'note': 'single-building owner', 'address': '2156 Cruger Avenue, Bronx, NY 10462'}, 'agent': {'name': 'HSC Management Corp., AMO', 'company': True, 'address': '5804A Tyndall Avenue, Bronx, NY 10471'}, 'registration': {'hpd_building_id': 60119, 'last_filed': '2025-10-08', 'record_url': 'https://hpdonline.nyc.gov/hpdonline/building/60119'}}


In [8]:
d = app.draft_message(p, "tenant", "2156 Cruger Avenue, Bronx, NY 10462")
print("To:", *d["recipients"], sep="\n  ")
print("Subject:", d["subject"], "\n")
print(d["body"])

To:
  2156 Cruger Avenue Apartment Corp, registered owner, 2156 Cruger Avenue, Bronx, NY 10462
  c/o HSC Management Corp., AMO, managing agent, 5804A Tyndall Avenue, Bronx, NY 10471
Subject: Request for a heating check before winter: 2156 Cruger Avenue, Bronx, NY 10462 

Dear 2156 Cruger Avenue Apartment Corp,

I am a resident of 2156 Cruger Avenue, Bronx, NY 10462 (BBL 2043190010), and I am writing before the cold weather to ask that the building's heating system be inspected and serviced.

HeatAhead, a public model built from New York City heat complaint and violation records, gives this building a very high (roughly 94–99%) modeled chance of at least one heat outage this winter, rank 323 of 63,727 buildings citywide, about 13.1 times the citywide median. City records show 127 heat complaints and 0 heat violations in the 2025-26 heat season; the most recent complaint on record was filed on 2026-10-08.

This is a statistical estimate, not a finding about the building's equipment. I wo

In [9]:
d = app.draft_message(p, "landlord", "2156 Cruger Avenue, Bronx, NY 10462")
print("To:", *d["recipients"], sep="\n  ")
print("Subject:", d["subject"], "\n")
print(d["body"])

To:
  2156 Cruger Avenue Apartment Corp, registered owner, 2156 Cruger Avenue, Bronx, NY 10462
  HSC Management Corp., AMO, managing agent, 5804A Tyndall Avenue, Bronx, NY 10471
Subject: Pre-winter heating self-audit: 2156 Cruger Avenue, Bronx, NY 10462 

Self-audit note for 2156 Cruger Avenue, Bronx, NY 10462 (BBL 2043190010), registered to 2156 Cruger Avenue Apartment Corp.

HeatAhead's model, built from New York City heat complaint and violation records, gives this building a very high (roughly 94–99%) modeled chance of at least one heat outage this winter, rank 323 of 63,727 buildings citywide, about 13.1 times the citywide median. The public record behind that estimate: 127 heat complaints and 0 heat violations in the 2025-26 heat season; the most recent complaint on record was filed on 2026-10-08.

Before the first cold spell:
1. Have a licensed contractor service the boiler and burner and test the low-water cutoff.
2. Confirm the fuel supply or gas account, and that this year's 

In [10]:
# both framings for all 63,727 ranked buildings, then the same checks on every draft
bbls = risk.bbl.tolist()
t0 = time.time()
payloads = [app.building_payload(b) for b in bbls]
t1 = time.time()
drafts = [app.draft_message(pl, role) for pl in payloads for role in app.DRAFT_ROLES]
t2 = time.time()
print(f"{len(payloads):,} payloads in {t1 - t0:.1f} s; {len(drafts):,} drafts in {t2 - t1:.1f} s "
      f"({len(drafts) / (t2 - t1):,.0f} drafts per second on {platform.machine()})")

import re
# an apartment number is a designator followed by a number ("Apt 4B", "#3A2", "Unit 12");
# company names such as "Kingsway Gardens Apt Corp" or "Five Unit Homes LLC" are fine
unit_rx = re.compile(r"#\s*\d|\b(?:apt|apartment|unit)\b\.?\s*#?\s*\d", re.I)
bad_unit = sum(bool(unit_rx.search(d["body"] + " ".join(d["recipients"]))) for d in drafts)
over_99 = sum("100%" in d["body"] for d in drafts)
system_signed = sum(d["body"].rstrip().endswith("HeatAhead") for d in drafts)
print("drafts with an apartment number:", bad_unit, "| quoting 100%:", over_99,
      "| signed by the system:", system_signed)
assert bad_unit == over_99 == system_signed == 0

63,727 payloads in 12.0 s; 127,454 drafts in 1.1 s (113,115 drafts per second on s390x)
drafts with an apartment number: 0 | quoting 100%: 0 | signed by the system: 0


## Result

- Section 2's table is the coverage the building page now shows: owner, managing agent and
  business address, with the HPD Online link for every registered building.
- `public/data/` was rebuilt here on s390x; copy it back and commit it with the code
  (`docs/VERCEL.md` section 2), then restart `heatahead-web` so the FULL app on this
  instance reads `bbl_contacts.csv` too.
- T5c (an AI-written draft) stays roadmap: it would replace `draft_message()` behind the
  same `/api/draft_message` response, keeping `note: "draft only"` and a human send step.